## 随机挑选细胞展示
该指导教程，只是因为获取的细胞数量太多，从已有的数据中随机挑选细胞进行UMAP结果展示

In [1]:
library(Seurat)
library(ggplot2)
library(patchwork)
library(dplyr)
library(Signac)
library(GenomicRanges)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following object is masked from ‘package:SeuratObject’:

    intersect


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, 

In [8]:
files<- list.files("/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data")
files

[1] "BRCA"  "CEAD"  "CESC"  "CRC"   "HNSCC" "OV"    "PDAC"  "SKCM"  "UCEC"

### 统一peak，并重新获取peak x cell的矩阵

In [9]:
setwd("/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/3_merge_data")

In [10]:
data_list <- setNames(
  lapply(files, function(f) readRDS(paste0(f,"/", f, "_annoAll.rds"))),
  gsub("\\.rds$", "", basename(files))  # 保留原来的名字作为 list 名
)

In [11]:
data_list

$BRCA
An object of class Seurat 
866810 features across 72587 samples within 4 assays 
Active assay: RNA (36601 features, 0 variable features)
 14 layers present: counts.1, counts.2, counts.3, counts.4, counts.5, counts.6, counts.7, counts.8, counts.9, counts.10, counts.11, counts.12, counts.13, counts.14
 3 other assays present: ATAC, peaks, SCT
 3 dimensional reductions calculated: pca, lsi, wnn.umap.unint

$CEAD
An object of class Seurat 
575830 features across 8156 samples within 4 assays 
Active assay: SCT (24690 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 3 other assays present: RNA, ATAC, peaks
 3 dimensional reductions calculated: pca, lsi, wnn.umap

$CESC
An object of class Seurat 
749687 features across 30100 samples within 4 assays 
Active assay: SCT (26699 features, 3000 variable features)
 3 layers present: counts, data, scale.data
 3 other assays present: RNA, ATAC, peaks
 3 dimensional reductions calculated: pca, lsi, wnn.umap

$CRC
An 

In [13]:
data_list <- lapply(data_list, function(x) {
  DefaultAssay(x) <- "peaks"
  return(x)
})

In [14]:
combined.peaks <- UnifyPeaks(object.list = data_list,mode = "reduce")

In [15]:
peakwidths <- width(combined.peaks)
combined.peaks <- combined.peaks[peakwidths  < 10000 & peakwidths > 20]

In [16]:
setwd("/mnt/netshare2/miaoyuanyuan/work/3_Cancer/1_cancer11/4_all_data/Pan_cancer")

In [18]:
combined.peaks

GRanges object with 836046 ranges and 0 metadata columns:
           seqnames            ranges strand
              <Rle>         <IRanges>  <Rle>
       [1]     chr1        9942-10597      *
       [2]     chr1       17240-17556      *
       [3]     chr1       79012-79595      *
       [4]     chr1       88097-88347      *
       [5]     chr1     180552-181627      *
       ...      ...               ...    ...
  [836042]     chrY 26660834-26661634      *
  [836043]     chrY 26663536-26664748      *
  [836044]     chrY 26669938-26671772      *
  [836045]     chrY 56677208-56678022      *
  [836046]     chrY 56685360-56685611      *
  -------
  seqinfo: 24 sequences from an unspecified genome; no seqlengths

### 从每个样本中随机挑选1000个细胞进行展示

In [ ]:
set.seed(2026) 
data_list_sub <- lapply(data_list, function(obj) {
  # 获取样本信息
  sample_ids <- unique(obj$sample)
  
  # 对每个样本下采样 1000 个细胞
  sampled_cells <- unlist(lapply(sample_ids, function(sid) {
    cells_in_sample <- Cells(obj)[obj$sample == sid]
    n_sample <- min(1000, length(cells_in_sample))
    sample(cells_in_sample, size = n_sample)
  }))
  
  # 返回下采样后的 Seurat 对象
  subset(obj, cells = sampled_cells)
})

In [ ]:
# 1. 加载并行计算包 (Seurat 和 Signac 自带的御用并行包)
library(future)

# 2. 突破默认的内存封锁线 (极其重要！)
# 默认情况下，R 只允许给子线程分配 500MB 的数据。
# 你的 combined.peaks 肯定极大，必须强行把上限拉高（这里设置为 100GB），否则绝对会报错！
options(future.globals.maxSize = 100 * 1024^3)

# 3. 召唤多线程 (开启 multisession)
# workers 代表你要调用的 CPU 核心数。
# 建议根据你服务器的闲置情况设置，通常 8 到 16 就能带来巨大的加速。
plan("multisession", workers = 16)

# 4. 运行你的计数代码
# 注意：代码完全不用改！Signac 一旦检测到你设置了 plan，
# 就会自动把几十万个 peaks 切成小块，分发给你刚才召唤的 16 个核心同时去扫描 Fragment 文件！
cat("多线程已就位，开始极速回溯计算 Counts...\n")
count.list <- lapply(data_list, function(obj) {
  FeatureMatrix(
    fragments = Fragments(obj),
    features = combined.peaks,
    cells = colnames(obj)
  )
})
save(count.list,file="count.list.rda")
# 5. 卸磨杀驴（极其良好的生信习惯）
# 算完之后，记得把多线程关掉，把这 16 个核心还给服务器，避免占用资源被其他同学拔网线。
plan("sequential")

In [20]:
annotation <- GetGRangesFromEnsDb(ensdb = EnsDb.Hsapiens.v86)
seqlevels(annotation) <- paste0("chr", seqlevels(annotation))

Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warning message in .merge_two_Seqinfo_objects(x, y):
"The 2 combined objects have no sequence levels in common. (Use
  suppressWarnings() to suppress this warning.)"
Warn

In [ ]:
library(Matrix)

# 假设 count.list 是稀疏矩阵列表，每个矩阵 peak × cell

num_peaks <- nrow(count.list[[1]])
total_cells <- numeric(num_peaks)

for (mat in count.list) {
  # 每个 peak 在多少细胞中有表达
  total_cells <- total_cells + Matrix::rowSums(mat > 0)
}

# 设置阈值，例如至少在 50 个细胞中有表达
keep_peaks <- which(total_cells > 50)

# 筛选 count.list
count.list.filtered <- lapply(count.list, function(mat) mat[keep_peaks, , drop = FALSE])

# 查看保留的 peak 数
length(keep_peaks)

In [ ]:
# ###################################### 
# 这里太慢了，于是手动合并，直接用cbind，行是features，列是cells，所以按列合并
# - 合并时,只对counts合并,包括RNA和ATAC的,以及metadata(有些列是独特的,因此直接用rbind会报错)
# - 注意,这里的RNA合并后还需要按照samples 进行 split,因为SCTransform需要根据单个样本进行归一化-测序深度等,它自动就是会对单个样本的counts进行处理

######################################## 

# cancer <- merge(x = obj.list[[1]], 
#                         y = obj.list[2:length(seurat_list)], 
#                         add.cell.ids = cancer_names, 
#                         project = "PanCancer_Atlas")

# rm(seurat_list)
# rm(seurat_list_n)
# gc() # 强制垃圾回收

# cancer

meta_list <- list()

for (i in seq_along(cancer_names)) {
  cancer <- cancer_names[i]
  
  # 1. 修改 Count 矩阵的列名 (例如变成 BRCA_AAACAGCCAAGGTCCT-1)
  colnames(count.list[[i]]) <- paste0(cancer, "_", colnames(count.list[[i]]))
  
  # 2. 提取并修改 Meta.data 的行名
  meta <- data_list[[i]]@meta.data
  rownames(meta) <- paste0(cancer, "_", rownames(meta))
  meta_list[[i]] <- meta
}

library(dplyr)
library(tibble)
big_meta <- lapply(meta_list, function(df) {
  # 提前把带有癌症前缀的行名变成一列，防止丢失
  as.data.frame(df) %>% rownames_to_column(var = "Cell_ID")
}) %>% 
  bind_rows() %>%  # 核心魔法：遇到不一致的列自动补 NA，完美融合！
  column_to_rownames(var = "Cell_ID")


genome(annotation) <- "hg38"

big_counts <- do.call(cbind, count.list)
chrom_assay <- CreateChromatinAssay(
  counts = big_counts,
  sep = c("-", "-"),  # Signac 默认的 peak 分隔符格式
  genome = 'hg38',
  annotation = annotation
)

rna_counts_list <- lapply(seq_along(data_list), function(i) {
  # 取出当前癌症的对象
  obj <- data_list[[i]]
  DefaultAssay(obj) <- "RNA"
  
  # 【极其关键的一步】：因为你是 Seurat V5，存在多个 counts layer
  # 必须先用 JoinLayers 把它们融合成一个完整的 counts 矩阵
  if (length(Layers(obj, search = "counts")) > 1) {
    obj <- JoinLayers(obj)
  }
  
  # 提取 RNA count 矩阵
  mat <- GetAssayData(obj, layer = "counts") 
  
  # 给细胞名加上与 ATAC 完全一样的癌症前缀，保证完美配对！
  colnames(mat) <- paste0(cancer_names[i], "_", colnames(mat))
  
  return(mat)
})
big_rna <- do.call(cbind, rna_counts_list)

PanCancer_TME <- CreateSeuratObject(
  counts = big_rna,
  assay = "RNA",
  meta.data = big_meta,
  project = "PanCancer_Multiome"
)

PanCancer_TME[["ATAC"]] <- chrom_assay   ## 实际上这里应该叫做peaks,后续会改

PanCancer_TME$pancancer_type<- sub("_.*", "", colnames(PanCancer_TME))

PanCancer_TME[["RNA"]] <- split(PanCancer_TME[["RNA"]], f = PanCancer_TME$sample)

saveRDS(PanCancer_TME,"./results/pancancer_merge.rds")

In [22]:
pan_cancer_obj <- SCTransform(pan_cancer_obj,vars.to.regress = c("nCount_RNA", 
                                                 "percent.mt"), return.only.var.genes = F)
pan_cancer_obj <- RunPCA(pan_cancer_obj)

Running SCTransform on assay: RNA

vst.flavor='v2' set. Using model with fixed slope and excluding poisson genes.

Calculating cell attributes from input UMI matrix: log_umi

Variance stabilizing transformation of count matrix of size 17094 by 654

Model formula is y ~ log_umi

Get Negative Binomial regression parameters per gene

Using 2000 genes, 654 cells

Found 64 outliers - those will be ignored in fitting/regularization step


Second step: Get residuals using fitted parameters for 17094 genes

Computing corrected count matrix for 17094 genes

Calculating gene attributes

Wall clock passed: Time difference of 11.63257 secs

Determine variable features

Regressing out nCount_RNA, percent.mt

Centering data matrix

Place corrected count matrix in counts slot

vst.flavor='v2' set. Using model with fixed slope and excluding poisson genes.

Calculating cell attributes from input UMI matrix: log_umi

Variance stabilizing transformation of count matrix of size 18525 by 1100

Model formul

In [23]:
DefaultAssay(pan_cancer_obj) <- "peaks"
pan_cancer_obj <- FindTopFeatures(pan_cancer_obj, min.cutoff = 5)
pan_cancer_obj <- RunTFIDF(pan_cancer_obj)
pan_cancer_obj<- RunSVD(pan_cancer_obj)

Performing TF-IDF normalization

Running SVD

Scaling cell embeddings



In [41]:
pan_cancer_obj <- FindMultiModalNeighbors(
  object = pan_cancer_obj,
  reduction.list = list("pca", "lsi"), 
  dims.list = list(1:30, 2:30),
  modality.weight.name = "RNA.weight",
  verbose = TRUE
)
pan_cancer_obj <- RunUMAP(
  object = pan_cancer_obj,
  nn.name = "weighted.nn",
    reduction.name = "wnn.umap", # 起了个新名字
  reduction.key = "wnnHUMAP_",
  verbose = TRUE
)

Calculating cell-specific modality weights

Finding 20 nearest neighbors for each modality.

Calculating kernel bandwidths

Warning message in FindMultiModalNeighbors(object = CEAD_SC, reduction.list = list("pca", :
"The number of provided modality.weight.name is not equal to the number of modalities. SCT.weight peaks.weight are used to store the modality weights"
Finding multimodal neighbors

Constructing multimodal KNN graph

Constructing multimodal SNN graph

13:13:19 UMAP embedding parameters a = 0.9922 b = 1.112

13:13:23 Commencing smooth kNN distance calibration using 1 thread
 with target n_neighbors = 20

13:13:30 Initializing from normalized Laplacian + noise (using RSpectra)

13:13:31 Commencing optimization for 200 epochs, with 1257352 positive edges

13:13:31 Using rng type: pcg

13:14:03 Optimization finished



In [ ]:
pan_cancer_obj$class<-pan_cancer_obj$celltype
pan_cancer_obj$class[pan_cancer_obj$class!="Tumor"]="Normal"

In [ ]:
pan_cancer_obj$celltype[pan_cancer_obj$celltype%in%c("Epithelial cells","Normal cells endometrium","Normal Epithelial cells","Normal Squamous cells")]="Epithelial cells"
pan_cancer_obj$celltype[pan_cancer_obj$celltype%in%c("Other","Unknown","UnKnown","Unkown")]="Unknown"

In [42]:
pan_cancer_obj <- FindClusters(
  pan_cancer_obj, 
  graph.name = "wsnn", # wsnn 是 FindMultiModalNeighbors 默认生成的联合图名称
  algorithm = 3,       # 匹配文献：3 代表 SLM 算法, algorithm = 4 是 Leiden。
  resolution = 0.1,
  verbose = FALSE
)

In [ ]:
pdf("pan_cancer_obj.umap_res_0.1.pdf", width = 8, height = 6)
DimPlot(pan_cancer_obj,group.by = "celltype", label = TRUE, repel = TRUE, reduction = "wnn.umap")
dev.off()

In [ ]:
pan_cancer_obj <- FindClusters(
  pan_cancer_obj, 
  graph.name = "wsnn", # wsnn 是 FindMultiModalNeighbors 默认生成的联合图名称
  algorithm = 3,       # 匹配文献：3 代表 SLM 算法, algorithm = 4 是 Leiden。
  resolution = 0.5,
  verbose = FALSE
)

In [ ]:
pdf("pan_cancer_obj.umap_res_0.5.pdf", width = 8, height = 6)
DimPlot(pan_cancer_obj,group.by = "celltype", label = TRUE, repel = TRUE, reduction = "wnn.umap")
dev.off()

In [ ]:
saveRDS(pan_cancer_obj,"pan_cancer_obj.rds")